# Self-attention by hand, and why we divide by √d

*A companion to the blog post “Self-attention: every word looks at every other word” (Part 4 of the Transformer on-ramp series).*

You will (1) compute attention for a tiny sentence step by step and (2) measure why the scores are divided by √d. Runs in a few seconds on the free Colab CPU.

In [ ]:
import torch
torch.manual_seed(0)

## 1. A tiny sentence, by hand
Four tokens, each a 4-number vector. Random projection matrices stand in for what a trained model would have learned, so the weights themselves are not meaningful. The *mechanics* are.

In [ ]:
tokens = ['The', 'animal', 'was', 'tired']
d = 4
X = torch.randn(4, d)                              # one row per token
Wq, Wk, Wv = (torch.randn(d, d) for _ in range(3))

Q, K, V = X @ Wq, X @ Wk, X @ Wv                   # queries, keys, values
scores  = Q @ K.T / d ** 0.5                       # how well each query matches each key
weights = scores.softmax(dim=-1)                   # rows sum to 1
out     = weights @ V                              # blend the values

print('weights (row = token asking, column = token it reads):')
print(weights.round(decimals=2))
print('row sums:', weights.sum(-1))
print('output shape:', out.shape)

## 2. Why divide by √d?
If each number in a query and key has spread 1, their dot product has spread about √d. Large scores push softmax towards a hard one-hot choice. We measure it with 2,000 random queries and 64 keys each.

In [ ]:
torch.manual_seed(0)
for d in (16, 64, 256, 1024):
    q = torch.randn(2000, d); k = torch.randn(2000, 64, d)
    s = torch.einsum('qd,qkd->qk', q, k)
    raw, scaled = s.softmax(-1), (s / d**0.5).softmax(-1)
    print(d, f'spread {s.std():.2f} (theory {d**0.5:.2f})',
          f'top weight raw {raw.max(-1).values.mean():.3f}',
          f'scaled {scaled.max(-1).values.mean():.3f}')

**What to notice.** Without scaling the biggest weight climbs towards 1 as d grows: the layer makes a near-hard choice before learning anything. With the √d division the numbers stay the same at every size (about 0.107, roughly 7 times the uniform value 1/64).

**Try it:** change 64 keys to 8 or 512 and see how the scaled value moves with the uniform value 1/(number of keys).

### Want the full guided version?

This notebook is a small taster from **Chapter 1 of the course *Generative AI and LLMs with Python*** (course-2-generative-ai), one of 4 world-class courses on [aibits.ecubeanalytics.com](https://aibits.ecubeanalytics.com/?utm_source=colab&utm_medium=notebook&utm_campaign=genai_ch01).

**The full guided lab, quizzes and projects are in the course.** Chapter 1 is part of the free first module:

- [Read Chapter 1 free (sign up)](https://aibits.ecubeanalytics.com/main/document/genai_gate.php?c=course-2-generative-ai&f=chapters/course-2-generative-ai/01-on-ramp-neurons-to-the-transformer.html&utm_source=colab&utm_medium=notebook&utm_campaign=genai_ch01)
- [See the course demo](https://aibits.ecubeanalytics.com/main/document/genai_public/GENAILLMPYTHON.html?utm_source=colab&utm_medium=notebook&utm_campaign=genai_ch01)
